In [7]:
# This is the template for the submission. You can develop your algorithm in a regular Python script and copy the code here for submission.

# TEAM NAME ON KAGGLE
# "EXAMPLE_GROUP"

# GROUP NUMBER
# "group_XX"

# TEAM MEMBERS (E-MAIL, LEGI, KAGGLE USERNAME):
# "examplestudent1@ethz.ch", "12-345-678", "eXampl3stdNtone" 
# "examplestudent2@ethz.ch", "12-345-679", "xXexamplestudent2Xx"
# "examplestudent3@ethz.ch", "12-345-670", "mhealth_student_98"

# Smartwatch Location

In [1]:
from os import listdir
from os.path import isfile, join
import re
import time
from pathlib import Path
from typing import Iterable

import joblib
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from dataclasses import dataclass, field
from typing import Mapping, Sequence
from scipy.signal import welch
from sklearn.metrics import f1_score

from mhealth_activity import Recording


WATCH_KEYS = ("ax", "ay", "az", "gx", "gy", "gz", "mx", "my", "mz", "temperature", "altitude")
AXIS_GROUPS = {
    "acc": ("ax", "ay", "az"),
    "gyr": ("gx", "gy", "gz"),
    "mag": ("mx", "my", "mz"),
}


def log(message: str) -> None:
    timestamp = time.strftime("%H:%M:%S")
    line = f"[{timestamp}] {message}"
    print(line, flush=True)



def parse_trace_id(path: Path) -> int:
    match = re.search(r"(\d{3})\.pkl$", path.name)
    if match is None:
        raise ValueError(f"Could not parse trace id from {path.name}")
    return int(match.group(1))


def downsample(values: Iterable[float], max_len: int = 3000) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if len(arr) <= max_len:
        return arr
    idx = np.linspace(0, len(arr) - 1, max_len).astype(int)
    return arr[idx]


def signal_features(values: np.ndarray, samplerate: float) -> dict[str, float]:
    names = (
        "mean",
        "std",
        "median",
        "q05",
        "q25",
        "q75",
        "q95",
        "min",
        "max",
        "range",
        "rms",
        "mad",
        "zcr",
        "dom_freq",
        "dom_power",
        "spec_entropy",
    )
    if len(values) == 0:
        return {name: 0.0 for name in names}

    mean = float(np.mean(values))
    feats = {
        "mean": mean,
        "std": float(np.std(values)),
        "median": float(np.median(values)),
        "q05": float(np.quantile(values, 0.05)),
        "q25": float(np.quantile(values, 0.25)),
        "q75": float(np.quantile(values, 0.75)),
        "q95": float(np.quantile(values, 0.95)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
        "rms": float(np.sqrt(np.mean(values**2))),
        "mad": float(np.mean(np.abs(values - mean))),
        "zcr": float(np.mean(np.diff(np.signbit(values)) != 0)) if len(values) > 1 else 0.0,
    }
    feats["range"] = feats["max"] - feats["min"]

    if len(values) > 32 and samplerate > 0:
        freqs, power = signal.welch(values - mean, fs=samplerate, nperseg=min(256, len(values)))
        power = np.maximum(power, 1e-12)
        feats["dom_freq"] = float(freqs[np.argmax(power[1:]) + 1] if len(power) > 1 else 0.0)
        feats["dom_power"] = float(np.max(power))
        power_share = power / np.sum(power)
        feats["spec_entropy"] = float(-(power_share * np.log(power_share)).sum())
    else:
        feats["dom_freq"] = 0.0
        feats["dom_power"] = 0.0
        feats["spec_entropy"] = 0.0

    return feats


def safe_corr(a: np.ndarray, b: np.ndarray) -> float:
    if len(a) < 4 or len(b) < 4:
        return 0.0
    corr = np.corrcoef(a, b)[0, 1]
    return 0.0 if not np.isfinite(corr) else float(corr)


def extract_watch_location_features(recording: Recording) -> dict[str, float]:
    features: dict[str, float] = {}
    if "ax" in recording.data:
        features["duration_s"] = float(recording.data["ax"].total_time)

    grouped_axes: dict[str, list[np.ndarray]] = {name: [] for name in AXIS_GROUPS}

    for key in WATCH_KEYS:
        if key not in recording.data:
            continue

        trace = recording.data[key]
        values = downsample(trace.values)
        for name, value in signal_features(values, trace.samplerate).items():
            features[f"{key}_{name}"] = value
        features[f"{key}_samplerate"] = float(trace.samplerate)
        features[f"{key}_gap"] = float(trace.max_update_gap)

        for prefix, axis_names in AXIS_GROUPS.items():
            if key in axis_names:
                max_len = 4000 if prefix != "mag" else 1500
                grouped_axes[prefix].append(downsample(trace.values, max_len=max_len))

    for prefix, axis_names in AXIS_GROUPS.items():
        axes = grouped_axes[prefix]
        if len(axes) != 3:
            continue

        usable_len = min(len(axis) for axis in axes)
        stacked = np.vstack([axis[:usable_len] for axis in axes])
        magnitude = np.sqrt((stacked**2).sum(axis=0))
        samplerate = float(np.mean([recording.data[name].samplerate for name in axis_names if name in recording.data]))

        for name, value in signal_features(downsample(magnitude), samplerate).items():
            features[f"{prefix}mag_{name}"] = value

        features[f"{prefix}_corr_xy"] = safe_corr(stacked[0], stacked[1])
        features[f"{prefix}_corr_xz"] = safe_corr(stacked[0], stacked[2])
        features[f"{prefix}_corr_yz"] = safe_corr(stacked[1], stacked[2])

    return features


def load_dataset(data_dir: Path, labeled: bool) -> tuple[pd.DataFrame, np.ndarray, np.ndarray | None, np.ndarray | None]:
    rows: list[dict[str, float]] = []
    ids: list[int] = []
    labels: list[int] = []
    groups: list[int] = []
    paths = sorted(data_dir.glob("*.pkl"))
    total = len(paths)
    start_time = time.time()

    for idx, path in enumerate(paths, start=1):
        recording = Recording(str(path))
        rows.append(extract_watch_location_features(recording))
        ids.append(parse_trace_id(path))
        if labeled:
            labels.append(int(recording.labels["watch_loc"]))
            groups.append(int(recording.labels["path_idx"]))
        if idx == 1 or idx % 25 == 0 or idx == total:
            elapsed = time.time() - start_time
            log(f"  Processed {idx}/{total} traces from {data_dir.name} in {elapsed:.1f}s")

    frame = pd.DataFrame(rows).replace([np.inf, -np.inf], np.nan)
    result = (
        frame,
        np.asarray(ids, dtype=int),
        np.asarray(labels, dtype=int) if labeled else None,
        np.asarray(groups, dtype=int) if labeled else None,
    )
    return result


def build_model() -> Pipeline:
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            (
                "classifier",
                ExtraTreesClassifier(
                    n_estimators=500,
                    max_depth=12,
                    min_samples_leaf=3,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=1,
                ),
            ),
        ]
    )


def align_feature_columns(features: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    aligned = features.reindex(columns=feature_columns, fill_value=np.nan)
    return aligned


def evaluate_model(features: pd.DataFrame, labels: np.ndarray, path_groups: np.ndarray) -> None:
    log("Evaluating watch-location model...")
    model = build_model()

    train_scores: list[float] = []
    stratified_scores: list[float] = []
    stratified_confusion = np.zeros((3, 3), dtype=int)
    stratified_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for train_idx, valid_idx in stratified_cv.split(features, labels):
        model.fit(features.iloc[train_idx], labels[train_idx])
        train_predictions = model.predict(features.iloc[train_idx])
        predictions = model.predict(features.iloc[valid_idx])
        train_scores.append(accuracy_score(labels[train_idx], train_predictions))
        stratified_scores.append(accuracy_score(labels[valid_idx], predictions))
        stratified_confusion += confusion_matrix(labels[valid_idx], predictions, labels=[0, 1, 2])

    grouped_scores: list[float] = []
    grouped_cv = GroupKFold(n_splits=5)
    for train_idx, valid_idx in grouped_cv.split(features, labels, groups=path_groups):
        model.fit(features.iloc[train_idx], labels[train_idx])
        predictions = model.predict(features.iloc[valid_idx])
        grouped_scores.append(accuracy_score(labels[valid_idx], predictions))

    model.fit(features, labels)
    final_train_predictions = model.predict(features)
    full_train_score = accuracy_score(labels, final_train_predictions)

    class_counts = pd.Series(labels).value_counts().sort_index().to_dict()
    per_class_recall = stratified_confusion.diagonal() / np.maximum(stratified_confusion.sum(axis=1), 1)

    log("Watch-location summary")
    log(f"  Training traces: {len(features)}")
    log(f"  Feature columns: {features.shape[1]}")
    log(f"  Class counts (0=wrist, 1=belt, 2=ankle): {class_counts}")
    log(f"  Full-train accuracy: {full_train_score:.4f}")
    log(f"  Mean fold training accuracy: {np.mean(train_scores):.4f} +/- {np.std(train_scores):.4f}")
    log(f"  Mean 5-fold validation accuracy: {np.mean(stratified_scores):.4f} +/- {np.std(stratified_scores):.4f}")
    log(f"  Mean 5-fold grouped-by-path validation accuracy: {np.mean(grouped_scores):.4f} +/- {np.std(grouped_scores):.4f}")
    log("  Stratified CV confusion matrix (rows=true, cols=pred):")
    log(str(stratified_confusion))
    log(
        "  Stratified CV per-class recall "
        f"(wrist, belt, ankle): {[round(float(x), 4) for x in per_class_recall]}"
    )


def train_watch_location_model(
    train_dir: Path,
    evaluate: bool = True,
    model_output: Path | None = None,
) -> dict[str, object]:
    log(f"Loading training traces from {train_dir} ...")
    train_x, _, train_y, train_groups = load_dataset(train_dir, labeled=True)

    if evaluate and train_y is not None and train_groups is not None:
        evaluate_model(train_x, train_y, train_groups)

    model = build_model()
    log("Training final model on all training traces...")
    model.fit(train_x, train_y)

    artifact = {
        "model": model,
        "feature_columns": list(train_x.columns),
        "label_map": {0: "wrist", 1: "belt", 2: "ankle"},
        "task": "watch_location",
    }

    if model_output is not None:
        model_output.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(artifact, model_output)
        log(f"Saved watch-location model to {model_output}")

    return artifact


def load_watch_location_model(model_path: Path) -> dict[str, object]:
    artifact = joblib.load(model_path)
    required_keys = {"model", "feature_columns", "task"}
    missing_keys = required_keys.difference(artifact.keys())
    if missing_keys:
        raise ValueError(f"Saved model at {model_path} is missing keys: {sorted(missing_keys)}")
    if artifact["task"] != "watch_location":
        raise ValueError(f"Saved model at {model_path} is not a watch-location model")
    log(f"Loaded watch-location model from {model_path}")
    return artifact


def predict_watch_locations_from_artifact(
    artifact: dict[str, object],
    test_dir: Path,
    output_csv: Path,
) -> pd.DataFrame:
    log(f"Loading test traces from {test_dir} ...")
    test_x, test_ids, _, _ = load_dataset(test_dir, labeled=False)
    feature_columns = artifact["feature_columns"]
    test_x = align_feature_columns(test_x, feature_columns)

    log("Predicting watch locations for test traces...")
    predicted_watch_locations = artifact["model"].predict(test_x).astype(int)

    predictions = pd.DataFrame(
        {
            "Id": test_ids,
            "watch_loc": predicted_watch_locations,
        }
    ).sort_values("Id")

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    predictions.to_csv(output_csv, index=False)
    log(f"Saved watch-location predictions to {output_csv}")
    return predictions


def predict_watch_locations(
    train_dir: Path,
    test_dir: Path,
    output_csv: Path,
    evaluate: bool,
    model_output: Path | None = None,
) -> None:
    artifact = train_watch_location_model(
        train_dir=train_dir,
        evaluate=evaluate,
        model_output=model_output,
    )
    predict_watch_locations_from_artifact(
        artifact=artifact,
        test_dir=test_dir,
        output_csv=output_csv,
    )

# Activity Prediction

## Inline Activity Recognition Code

In [2]:
ACTIVITY_ORDER = ("standing", "walking", "running", "cycling")
WATCH_MOTION_GROUPS = {
    "watch_acc": ("ax", "ay", "az"),
    "watch_gyr": ("gx", "gy", "gz"),
    "watch_mag": ("mx", "my", "mz"),
}

DEFAULT_WINDOW_S = 15.0
DEFAULT_HOP_S = 5.0
DEFAULT_MIN_ACTIVITY_S = 60.0


def normalize_activities(raw_value) -> list[str]:
    if isinstance(raw_value, Mapping):
        return [name for name in ACTIVITY_ORDER if bool(raw_value.get(name, False))]

    if isinstance(raw_value, (list, tuple, set, np.ndarray)):
        parsed = []
        for item in raw_value:
            if isinstance(item, str):
                label = item.strip().lower()
                if label in ACTIVITY_ORDER:
                    parsed.append(label)
            elif isinstance(item, (int, np.integer)) and 0 <= int(item) < len(ACTIVITY_ORDER):
                parsed.append(ACTIVITY_ORDER[int(item)])
        return [name for name in ACTIVITY_ORDER if name in set(parsed)]

    return []


def summarize_signal(values: Sequence[float], samplerate: float) -> dict[str, float]:
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        raise ValueError("Signal summary requires at least one sample.")

    centered = values - np.mean(values)
    freqs, power = welch(centered, fs=samplerate, nperseg=min(256, len(values)))
    dom_idx = np.argmax(power[1:]) + 1 if len(power) > 1 else 0

    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
        "rms": float(np.sqrt(np.mean(values ** 2))),
        "iqr": float(np.quantile(values, 0.75) - np.quantile(values, 0.25)),
        "p10": float(np.quantile(values, 0.10)),
        "p90": float(np.quantile(values, 0.90)),
        "dom_freq": float(freqs[dom_idx]) if len(freqs) else 0.0,
        "dom_power": float(power[dom_idx]) if len(power) else 0.0,
        "spec_energy": float(power.sum()) if len(power) else 0.0,
    }


def extract_window_feature_rows(
    recording: Recording,
    motion_groups: Mapping[str, Sequence[str]] = WATCH_MOTION_GROUPS,
    window_s: float = DEFAULT_WINDOW_S,
    hop_s: float = DEFAULT_HOP_S,
) -> pd.DataFrame:
    base_keys = ("ax", "ay", "az")
    if not all(key in recording.data for key in base_keys):
        return pd.DataFrame()

    base_fs = float(np.mean([recording.data[key].samplerate for key in base_keys]))
    base_len = min(len(recording.data[key].values) for key in base_keys)
    window_n = max(int(window_s * base_fs), 1)
    hop_n = max(int(hop_s * base_fs), 1)

    if base_len < window_n:
        return pd.DataFrame()

    rows: list[dict[str, float]] = []
    for start_idx in range(0, base_len - window_n + 1, hop_n):
        end_idx = start_idx + window_n
        row = {
            "start_s": start_idx / base_fs,
            "end_s": end_idx / base_fs,
            "mid_s": (start_idx + end_idx) / (2 * base_fs),
            "window_s": float(window_s),
            "hop_s": float(hop_s),
        }

        for prefix, keys in motion_groups.items():
            if not all(key in recording.data for key in keys):
                continue

            usable_len = min(len(recording.data[key].values) for key in keys)
            if end_idx > usable_len:
                continue

            stacked = np.vstack(
                [recording.data[key].values[start_idx:end_idx].astype(float) for key in keys]
            )
            magnitude = np.sqrt((stacked ** 2).sum(axis=0))
            samplerate = float(np.mean([recording.data[key].samplerate for key in keys]))

            for feature_name, value in summarize_signal(magnitude, samplerate).items():
                row[f"{prefix}_{feature_name}"] = value

        rows.append(row)

    return pd.DataFrame(rows)


def _bridge_short_gaps(mask: Sequence[bool], max_gap_windows: int) -> np.ndarray:
    mask = np.asarray(mask, dtype=bool).copy()
    if max_gap_windows <= 0:
        return mask

    n_samples = len(mask)
    idx = 0
    while idx < n_samples:
        if mask[idx]:
            idx += 1
            continue

        gap_end = idx
        while gap_end < n_samples and not mask[gap_end]:
            gap_end += 1

        if idx > 0 and gap_end < n_samples and (gap_end - idx) <= max_gap_windows:
            mask[idx:gap_end] = True

        idx = gap_end

    return mask


def _smooth_labels(labels: Sequence[str]) -> np.ndarray:
    labels = np.asarray(labels, dtype=object)
    if len(labels) < 3:
        return labels.copy()

    smoothed = labels.copy()
    for idx in range(1, len(labels) - 1):
        triad = labels[idx - 1 : idx + 2]
        values, counts = np.unique(triad, return_counts=True)
        smoothed[idx] = values[np.argmax(counts)]
    return smoothed


def _longest_positive_run(mask: Sequence[bool], hop_s: float, window_s: float) -> float:
    best = 0.0
    current = 0
    for flag in mask:
        if flag:
            current += 1
            best = max(best, window_s + max(current - 1, 0) * hop_s)
        else:
            current = 0
    return best


def _activity_gap_windows(activity: str, hop_s: float) -> int:
    if activity in {"walking", "running"}:
        return int(np.floor(8.0 / hop_s))
    return 1


@dataclass
class ActivityRecognizer:
    classifier: ExtraTreesClassifier
    feature_columns: list[str]
    feature_medians: pd.Series
    motion_groups: Mapping[str, Sequence[str]] = field(default_factory=lambda: WATCH_MOTION_GROUPS)
    window_s: float = DEFAULT_WINDOW_S
    hop_s: float = DEFAULT_HOP_S
    min_activity_s: float = DEFAULT_MIN_ACTIVITY_S

    def label_windows(self, recording: Recording, watch_loc: int | None = None) -> pd.DataFrame:
        window_df = extract_window_feature_rows(
            recording,
            motion_groups=self.motion_groups,
            window_s=self.window_s,
            hop_s=self.hop_s,
        )
        if window_df.empty:
            return window_df
        if watch_loc is not None and not window_df.empty:
            window_df["watch_loc"] = int(watch_loc)

        x = window_df.reindex(columns=self.feature_columns).fillna(self.feature_medians)
        proba = self.classifier.predict_proba(x)
        classes = np.asarray(self.classifier.classes_)
        raw_labels = classes[np.argmax(proba, axis=1)]
        pred_labels = _smooth_labels(raw_labels)

        labeled = window_df.copy()
        labeled["raw_pred_label"] = raw_labels
        labeled["pred_label"] = pred_labels

        for activity_name in ACTIVITY_ORDER:
            labeled[f"prob_{activity_name}"] = 0.0

        for class_idx, class_name in enumerate(classes):
            labeled[f"prob_{class_name}"] = proba[:, class_idx]

        return labeled

    def summarize_labeled_windows(self, labeled_window_df: pd.DataFrame) -> pd.DataFrame:
        if labeled_window_df.empty:
            return pd.DataFrame(
                {
                    "activity": list(ACTIVITY_ORDER),
                    "longest_run_s": [0.0] * len(ACTIVITY_ORDER),
                    "meets_60s_rule": [False] * len(ACTIVITY_ORDER),
                }
            )

        hop_s = float(labeled_window_df["hop_s"].iloc[0])
        window_s = float(labeled_window_df["window_s"].iloc[0])

        rows = []
        for activity_name in ACTIVITY_ORDER:
            raw_mask = labeled_window_df["pred_label"].to_numpy() == activity_name
            bridged_mask = _bridge_short_gaps(
                raw_mask,
                max_gap_windows=_activity_gap_windows(activity_name, hop_s),
            )
            longest_run_s = _longest_positive_run(bridged_mask, hop_s=hop_s, window_s=window_s)
            rows.append(
                {
                    "activity": activity_name,
                    "longest_run_s": float(longest_run_s),
                    "meets_60s_rule": bool(longest_run_s >= self.min_activity_s),
                }
            )

        return pd.DataFrame(rows)

    def predict_activities(self, recording: Recording, watch_loc: int | None = None) -> dict[str, bool]:
        summary_df = self.summarize_labeled_windows(
            self.label_windows(recording, watch_loc=watch_loc)
        )
        return {
            row.activity: bool(row.meets_60s_rule)
            for row in summary_df.itertuples(index=False)
        }


def _trace_id_from_path(path: Path) -> int:
    return int(path.stem.split("_")[-1])


def build_activity_recognizer(
    train_dir: Path | str,
    motion_groups: Mapping[str, Sequence[str]] = WATCH_MOTION_GROUPS,
    window_s: float = DEFAULT_WINDOW_S,
    hop_s: float = DEFAULT_HOP_S,
    min_activity_s: float = DEFAULT_MIN_ACTIVITY_S,
    standing_fraction: float = 0.15,
    random_state: int = 42,
    n_estimators: int = 120,
    min_samples_leaf: int = 2,
) -> ActivityRecognizer:
    train_dir = Path(train_dir)
    metadata_rows = []
    window_cache: dict[str, pd.DataFrame] = {}
    all_feature_names: set[str] = set()

    for path in sorted(train_dir.glob("*.pkl")):
        recording = Recording(str(path))
        activities = normalize_activities((recording.labels or {}).get("activities", []))
        metadata_rows.append(
            {
                "file": path.name,
                "trace_id": _trace_id_from_path(path),
                "n_activities": len(activities),
                **{activity: activity in activities for activity in ACTIVITY_ORDER},
            }
        )

        window_df = extract_window_feature_rows(
            recording,
            motion_groups=motion_groups,
            window_s=window_s,
            hop_s=hop_s,
        )
        if not window_df.empty:
            window_df["watch_loc"] = int(recording.labels["watch_loc"])
        window_cache[path.name] = window_df
        all_feature_names.update(
            column
            for column in window_df.columns
            if column not in {"start_s", "end_s", "mid_s", "window_s", "hop_s"}
        )

    metadata_df = pd.DataFrame(metadata_rows).sort_values("trace_id").reset_index(drop=True)
    single_label_df = metadata_df.loc[metadata_df["n_activities"] == 1].copy()
    single_label_df["activity_label"] = single_label_df[list(ACTIVITY_ORDER)].idxmax(axis=1)

    training_parts = []
    for row in single_label_df.itertuples(index=False):
        window_df = window_cache[row.file]
        if window_df.empty:
            continue
        part = window_df.copy()
        part["file"] = row.file
        part["activity_label"] = row.activity_label
        training_parts.append(part)

    standing_subset = metadata_df.loc[metadata_df["standing"]].copy()
    for row in standing_subset.itertuples(index=False):
        window_df = window_cache[row.file]
        if window_df.empty:
            continue

        motion_score = window_df["watch_acc_std"] + 0.3 * window_df["watch_gyr_std"]
        keep_n = max(1, int(np.ceil(standing_fraction * len(window_df))))
        part = window_df.loc[motion_score.nsmallest(keep_n).index].copy()
        part["file"] = row.file
        part["activity_label"] = "standing"
        training_parts.append(part)

    if not training_parts:
        raise RuntimeError("No training windows were extracted for activity recognition.")

    training_df = pd.concat(training_parts, ignore_index=True)
    feature_columns = sorted(all_feature_names)
    feature_medians = training_df.reindex(columns=feature_columns).median(numeric_only=True)
    x_train = training_df.reindex(columns=feature_columns).fillna(feature_medians)
    y_train = training_df["activity_label"]

    classifier = ExtraTreesClassifier(
        n_estimators=n_estimators,
        min_samples_leaf=min_samples_leaf,
        class_weight="balanced",
        random_state=random_state,
        n_jobs=1,
    )
    classifier.fit(x_train, y_train)

    return ActivityRecognizer(
        classifier=classifier,
        feature_columns=feature_columns,
        feature_medians=feature_medians,
        motion_groups=motion_groups,
        window_s=window_s,
        hop_s=hop_s,
        min_activity_s=min_activity_s,
    )


def evaluate_activity_recognizer(
    recognizer: ActivityRecognizer,
    data_dir: Path | str,
) -> pd.DataFrame:
    data_dir = Path(data_dir)
    rows = []
    for path in sorted(data_dir.glob("*.pkl")):
        recording = Recording(str(path))
        truth_activities = normalize_activities((recording.labels or {}).get("activities", []))
        truth = {activity: activity in truth_activities for activity in ACTIVITY_ORDER}
        pred = recognizer.predict_activities(recording)

        row = {"file": path.name}
        for activity in ACTIVITY_ORDER:
            row[f"true_{activity}"] = bool(truth[activity])
            row[f"pred_{activity}"] = bool(pred[activity])
        rows.append(row)

    return pd.DataFrame(rows)


def activity_f1_summary(prediction_df: pd.DataFrame) -> pd.Series:
    scores = {
        activity: f1_score(
            prediction_df[f"true_{activity}"],
            prediction_df[f"pred_{activity}"],
        )
        for activity in ACTIVITY_ORDER
    }
    scores["macro"] = float(np.mean([scores[activity] for activity in ACTIVITY_ORDER]))
    return pd.Series(scores, dtype=float)


# Final submission

In [3]:
train_dir = Path("data/train")
test_dir = Path("data/test")

filenames = sorted(test_dir.glob("*.pkl"))

watch_location_artifact = train_watch_location_model(
    train_dir=train_dir,
    evaluate=True,
    model_output=None,
)

activity_recognizer = build_activity_recognizer(train_dir)
joblib.dump(activity_recognizer, 'group10_model_activity_nini.joblib')

# Optional: save the trained model artifact as a .joblib file.
#joblib.dump(watch_location_artifact, 'group10_model_watchloc.joblib')

# Optional alternative: load an already trained watch-location model instead of retraining.
# watch_location_model_path = Path('group10_model_watchloc.joblib')
# watch_location_artifact = load_watch_location_model(watch_location_model_path)


[13:02:57] Loading training traces from data\train ...
[13:02:57]   Processed 1/396 traces from train in 0.5s
[13:03:13]   Processed 25/396 traces from train in 16.0s
[13:03:29]   Processed 50/396 traces from train in 31.8s
[13:03:46]   Processed 75/396 traces from train in 49.0s
[13:04:00]   Processed 100/396 traces from train in 63.0s
[13:04:15]   Processed 125/396 traces from train in 78.3s
[13:04:33]   Processed 150/396 traces from train in 96.1s
[13:04:48]   Processed 175/396 traces from train in 110.7s
[13:05:02]   Processed 200/396 traces from train in 125.0s
[13:05:19]   Processed 225/396 traces from train in 142.3s
[13:05:37]   Processed 250/396 traces from train in 159.7s
[13:05:52]   Processed 275/396 traces from train in 175.3s
[13:06:09]   Processed 300/396 traces from train in 192.2s
[13:06:25]   Processed 325/396 traces from train in 207.9s
[13:06:40]   Processed 350/396 traces from train in 223.4s
[13:06:56]   Processed 375/396 traces from train in 238.8s
[13:07:10]   P

['group10_model_activity_nini.joblib']

In [4]:
import sklearn
print(sklearn.__version__)

1.6.1


In [11]:
# Loop through all filenames to process recordings
submission = []
for path in filenames:
    recording = Recording(str(path))
    id = parse_trace_id(path)

    # Placeholder for the algorithm to process the recording
    # Implement the logic to infer watch location, path index, step count,
    # and activities (standing, walking, running, cycling) here.
    # Ensure your algorithm is tolerant to missing data and does not crash
    # when optional smartphone data traces are missing.

    path_idx = -1  # Integer, path in {0, 1, 2, 3, 4}

    watch_loc_features = pd.DataFrame([extract_watch_location_features(recording)])
    watch_loc_features = align_feature_columns(
        watch_loc_features,
        watch_location_artifact['feature_columns'],
    )
    watch_loc = int(watch_location_artifact['model'].predict(watch_loc_features)[0])  # Integer, 0: left wrist, 1: belt, 2: right ankle


    predicted_activities = activity_recognizer.predict_activities(recording, watch_loc=watch_loc)

    standing = bool(predicted_activities["standing"])
    walking = bool(predicted_activities["walking"])
    running = bool(predicted_activities["running"])
    cycling = bool(predicted_activities["cycling"])
    step_count = -1  # Integer, number of steps, must be provided for each recording

    predictions = {
        'Id': id, 
        'watch_loc': watch_loc, 
        'path_idx': path_idx,
        'standing': standing,
        'walking': walking,
        'running': running,
        'cycling': cycling,
        'step_count': step_count
        }

    submission.append(predictions)


In [12]:
# Write the predicted values into a .csv file to then upload the .csv file to Kaggle
# When cross-checking the .csv file on your computer, we recommend using a text editor and NOT excel so that the results are displayed correctly
# IMPORTANT: Do NOT change the name of the columns of the .csv file ("Id", "watch_loc", "path_idx", "standing", "walking", "running", "cycling", "step_count")
submission_df = pd.DataFrame(submission, columns=['Id', 'watch_loc', 'path_idx', 'standing', 'walking', 'running', 'cycling', 'step_count']).sort_values('Id')
submission_df.to_csv('submission_loc_act.csv', index=False)  # Saves the submission file in the current working directory
#submission_df.to_csv('/kaggle/working/submission.csv', index=False)